# Cardio Catch Disease — Pipeline Walkthrough

> Cardiovascular risk screening from a routine examination.

This notebook walks through the production pipeline using the modules in `src/`. The model is loaded from the serialized artifact produced by `python -m src.pipeline`.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_columns", 50)

## 1. Data

Load the versioned sample and inspect it.

In [2]:
df = pd.read_csv(config.SAMPLE_PATH, sep=config.CSV_SEP)
print(df.shape)
df.head()

(6000, 13)


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,28427,19713,1,165,70.0,120,80,1,1,0,0,1,0
1,69891,14549,2,181,90.0,110,70,1,1,0,1,0,0
2,93324,22699,1,166,55.0,130,80,3,3,0,0,1,0
3,92040,23169,2,168,70.0,120,80,1,1,1,1,1,0
4,63785,14600,1,158,59.0,110,60,1,1,0,0,1,0


## 2. Preprocessing

The same transform used in training and serving.

In [3]:
from src.preprocessing import Preprocessor
X, y = Preprocessor().run(df)
print("features:", X.shape, "| disease rate:", round(y.mean(),4))
X.head()

features: (6000, 11) | disease rate: 0.4997


,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active
0,19713,1,165,70.0,120,80,1,1,0,0,1
1,14549,2,181,90.0,110,70,1,1,0,1,0
2,22699,1,166,55.0,130,80,3,3,0,0,1
3,23169,2,168,70.0,120,80,1,1,1,1,1
4,14600,1,158,59.0,110,60,1,1,0,0,1


## 3. Model and evaluation

Metrics from the serialized model card.

In [4]:
card = json.loads(Path(config.MODEL_CARD_PATH).read_text())
print(json.dumps(card, indent=2)[:1800])

{
  "schema_version": "1.0",
  "trained_at": "2026-06-14T14:24:11+00:00",
  "dataset": "sulianova/cardiovascular-disease-dataset",
  "data_sha256": "21a705d23381b0dfd6a6416da701b490744f1fc3b47e9ff3db3968c420ffa10c",
  "target": "cardio",
  "problem": "cardiovascular disease screening (balanced binary classification)",
  "best_model": "hist_gb",
  "best_params": {
    "clf__max_leaf_nodes": 63,
    "clf__max_iter": 400,
    "clf__max_depth": 3,
    "clf__learning_rate": 0.0608997324441945,
    "clf__l2_regularization": 1.0
  },
  "cv_selection": [
    {
      "model": "hist_gb",
      "roc_auc_mean": 0.8029232773133135,
      "roc_auc_std": 0.0024894615258046994
    },
    {
      "model": "logreg",
      "roc_auc_mean": 0.7939424097421266,
      "roc_auc_std": 0.001372607741189573
    },
    {
      "model": "random_forest",
      "roc_auc_mean": 0.7743865190376702,
      "roc_auc_std": 0.0026189114968609086
    }
  ],
  "baseline": {
    "model": "LogisticRegression",
    "roc_auc": 0

## 4. Prediction

The serving contract on a representative input.

In [5]:
from src.predict import Predictor
pred = Predictor()
rec = {"age":20000,"gender":2,"height":170,"weight":90,"ap_hi":150,"ap_lo":95,"cholesterol":3,"gluc":1,"smoke":1,"alco":0,"active":0}
s = pred.score_one(rec)
print("risk:", round(s,4), "|", pred.decision(s))

risk: 0.89 | elevated cardiovascular risk


## Reproduce

Run the full pipeline end to end:

```
python -m src.pipeline
```